# Prompt chaining

This pattern breaks down large, complex tasks into smaller and more manageable problems. It's also known as Pipeline pattern and it's similar in nature to traditional orchestration: the output generated by one prompt is passed as input to the next promt in the chain.

In [4]:
#First, prepare your environment
!uv pip install flyte openai

Using Python 3.12.12 environment at: /Users/davidmirror/code/agentic-patterns-on-v2/.venv
Audited 2 packages in 33ms


### Configure the execution environment

Configure your connection to the Flyte cluster. This tells Flyte where to run your workflows and how to build container images.

**Configuration Options:**
- `endpoint`: Your Flyte cluster URL
- `org`: Your organization name
- `project`: Project to organize workflows
- `domain`: Environment (development, staging, production)
- `image_builder`: Use "remote" to build images on the cluster (no local Docker required)

In [9]:
import flyte
flyte.init(
    endpoint="https://<MY_TENANT_HOST>",  # Your Union cluster URL
    org="<your-Union-org-name>",                                     # Your organization
    project="flytesnacks",                               # Your project name
    domain="development",                           # Environment: development/staging/production
    image_builder="remote",                         # Build images on cluster (no local Docker needed)
    auth_type="DeviceFlow",
)

Create a secret in Flyte for the API key (only done once):

In [11]:

!flyte create secret OPENAI_API_KEY --value sk-proj-...

- Flyte requires you to define a `TaskEnvironment`, an object that encapsulates the configuration your task will use during execution, including `Secrets`.
- Flyte allows you to [tune compute resources](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/resources/) (GPU, CPU, memory, ephemeral storage) via `Resources`.   
- [Caching](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/caching/) is enabled to save cost and drive down execution times for subsequent runs by avoiding LLM calls with the same inputs. 
- Flyte enables you to declare the desired configuration and contents of the [container image](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/container-images/) and have the image built automatically.

In [ ]:
import asyncio
import flyte
from flyte import TaskEnvironment, Resources

env = TaskEnvironment(
    name="llm_specs_env",
    image=flyte.Image.from_debian_base().with_pip_packages("openai"),
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")]
)


The two-promt chain is implemented as two tasks where each task will run remotely on a container (or locally in a Python process) with its own compute resources.

This example uses the `openai` API and Python to:  

- Prompt 1: extract technical specs from text_input:


In [5]:


@env.task
async def extract_specs(text_input: str) -> str:
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"Extract the technical specifications from:\n\n{text_input}"}],
    )
    return response.choices[0].message.content or ""


- Prompt 2: transform those specs into a JSON-like structure.

In [8]:
@env.task
async def specs_to_json(specifications: str) -> str:
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"Transform into JSON with 'cpu', 'memory', 'storage':\n\n{specifications}"}],
    )
    return response.choices[0].message.content or ""


In Flyte V2 you can have tasks calling other tasks, giving you the flexibility to write orchestration logic in pure Python.

In [ ]:
@env.task
async def process_description(text_input: str) -> str:
    specs = await extract_specs(text_input)
    return await specs_to_json(specs)

**Rule of thumb:** Use this pattern when a task is too complex for a single prompt, involves multiple distinct processing stages, requires interaction with external tools between steps, or when building Agentic systems that need to perform multi-step reasoning and maintain state.

## Scaling the pattern

- Wrap the LLM calls with `@flyte.trace` if you want fine-grained traces and checkpointing around the LLM steps, so failed runs can resume without redoing all work. [Learn more](https://www.union.ai/docs/v2/byoc/user-guide/task-programming/traces/)
- Use task-level `retries` for transient LLM/API errors, and `timeout` to guard against hung requests. [Learn more](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/retries-and-timeouts/)
- Make it batchable: add a task that handles many product descriptions in parallel, using async patterns:


In [ ]:
@env.task
@env.task
async def process_descriptions_batch(texts: list[str]) -> list[str]:
    sem = asyncio.Semaphore(20)
    async def _one(t: str) -> str:
        async with sem:
            return await process_description(t)
    tasks = [_one(t) for t in texts]
    return list(await asyncio.gather(*tasks))